# Mini Project — COVID-19 India EDA (Revised Version)

**Goal:** Perform a cleaner and more focused Exploratory Data Analysis of India's COVID-19 state-wise data.

### What is changed from the original project?
- **Time period:** 1 May 2020 to 6 August 2020 instead of the full dataset.
- **States:** Focus on **Maharashtra, Tamil Nadu, Delhi, Karnataka, Andhra Pradesh, Kerala, Uttar Pradesh, and West Bengal**.
- **Analysis:** Focuses on growth, active cases, recovery/death rates, and state comparisons.
- **Problems fixed:** safer numeric conversion, consistent state names, duplicate handling, zero-division protection, correct daily-case aggregation, removal of misleading "least affected" ranking, and dynamic conclusions instead of hard-coded figures.

**Data source:** `imdevskp/covid-19-india-data` (state-wise India COVID-19 case data, sourced from historical public health reporting).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 20)

print("Libraries loaded successfully.")

## 2. Load the Data

The project uses the same historical state-level source, but the analysis window and state subset are changed.

In [ ]:
URL = "https://raw.githubusercontent.com/imdevskp/covid-19-india-data/master/complete.csv"

raw = pd.read_csv(URL)

print("Raw shape:", raw.shape)
raw.head()

## 3. Inspect the Raw Data

In [ ]:
print("Columns:")
print(raw.columns.tolist())

print("\nData types:")
print(raw.dtypes)

print("\nMissing values:")
print(raw.isna().sum())

print("\nUnique state/UT count:", raw["Name of State / UT"].nunique())

## 4. Clean the Data — with fixes

The original notebook had several issues that could affect the analysis:

1. Text values in `Death` needed conversion.
2. State names had inconsistent spellings/prefixes.
3. Dropping duplicate `(date, state)` rows can silently discard information if duplicates are not identical.
4. National daily cases were calculated using a difference of cumulative totals after cleaning; this can hide reporting irregularities.
5. Recovery/death rates can produce invalid values when confirmed cases are zero.
6. Ranking the "least affected" states is not a meaningful epidemiological conclusion because population size, testing, reporting and exposure differ.

Here we clean conservatively and explicitly validate duplicates.

In [ ]:
cases = raw.copy()

# Standardize column names
cases.columns = [
    "date", "state", "lat", "long", "confirmed",
    "deaths", "cured", "new_cases", "new_deaths", "new_recovered"
]

# Convert date and numeric columns safely
cases["date"] = pd.to_datetime(cases["date"], errors="coerce")

numeric_cols = [
    "confirmed", "deaths", "cured",
    "new_cases", "new_deaths", "new_recovered"
]
for col in numeric_cols:
    cases[col] = pd.to_numeric(cases[col], errors="coerce")

# Remove rows with unusable dates/state names
cases = cases.dropna(subset=["date", "state"]).copy()

# Fill missing numeric reporting fields with 0
cases[numeric_cols] = cases[numeric_cols].fillna(0)

# Standardize historical state/UT spellings
name_fix = {
    "Telangana***": "Telangana",
    "Telengana": "Telangana",
    "Union Territory of Jammu and Kashmir": "Jammu and Kashmir",
    "Union Territory of Ladakh": "Ladakh",
    "Union Territory of Chandigarh": "Chandigarh",
}
cases["state"] = cases["state"].replace(name_fix)

# Check duplicates BEFORE removing them
duplicate_rows = cases.duplicated(subset=["date", "state"], keep=False)
print("Rows involved in (date, state) duplicates:", duplicate_rows.sum())

# If duplicate records exist, aggregate them rather than arbitrarily dropping rows.
# For cumulative fields, use max; for daily fields, use sum.
cases = (
    cases.groupby(["date", "state"], as_index=False)
    .agg({
        "lat": "first",
        "long": "first",
        "confirmed": "max",
        "deaths": "max",
        "cured": "max",
        "new_cases": "sum",
        "new_deaths": "sum",
        "new_recovered": "sum"
    })
)

# Derived metric
cases["active"] = cases["confirmed"] - cases["deaths"] - cases["cured"]

# Guard against small data-quality inconsistencies
cases["active"] = cases["active"].clip(lower=0)

cases = cases.sort_values(["state", "date"]).reset_index(drop=True)

print("Clean shape:", cases.shape)
print("Clean states:", cases["state"].nunique())
print("Remaining duplicate keys:", cases.duplicated(["date", "state"]).sum())

## 5. Select a New Time Period and State Group

This version studies a focused period in the first wave and compares eight selected states rather than all states.

In [ ]:
START_DATE = pd.Timestamp("2020-05-01")
END_DATE = pd.Timestamp("2020-08-06")

selected_states = [
    "Maharashtra",
    "Tamil Nadu",
    "Delhi",
    "Karnataka",
    "Andhra Pradesh",
    "Kerala",
    "Uttar Pradesh",
    "West Bengal"
]

analysis = cases[
    cases["date"].between(START_DATE, END_DATE)
    & cases["state"].isin(selected_states)
].copy()

print("Analysis period:", START_DATE.date(), "to", END_DATE.date())
print("States:", selected_states)
print("Rows:", len(analysis))

## 6. Build Reliable Daily and Cumulative Metrics

Instead of calculating daily national cases only by differencing a cumulative total, we use the reported `new_cases` field and aggregate it across the selected states. This preserves the dataset's daily reporting information.

In [ ]:
daily = (
    analysis.groupby("date", as_index=False)
    [["new_cases", "new_deaths", "new_recovered"]]
    .sum()
)

national_selected = (
    analysis.groupby("date", as_index=False)
    [["confirmed", "deaths", "cured", "active"]]
    .sum()
)

national_selected["recovery_rate"] = np.where(
    national_selected["confirmed"] > 0,
    national_selected["cured"] / national_selected["confirmed"] * 100,
    np.nan
)

national_selected["death_rate"] = np.where(
    national_selected["confirmed"] > 0,
    national_selected["deaths"] / national_selected["confirmed"] * 100,
    np.nan
)

print("Daily metric range:")
display(daily.head())

print("\nNational selected-state snapshot:")
display(national_selected.tail())

## 7. Latest State Snapshot

Rates are calculated only when a state has confirmed cases. For fairer rate comparisons, a minimum case threshold can also be applied.

In [ ]:
latest_date = analysis["date"].max()
latest = analysis[analysis["date"] == latest_date].copy()

latest["recovery_rate"] = np.where(
    latest["confirmed"] > 0,
    latest["cured"] / latest["confirmed"] * 100,
    np.nan
)

latest["death_rate"] = np.where(
    latest["confirmed"] > 0,
    latest["deaths"] / latest["confirmed"] * 100,
    np.nan
)

latest = latest.sort_values("confirmed", ascending=False).reset_index(drop=True)

display(latest[
    ["state", "confirmed", "active", "cured", "deaths",
     "recovery_rate", "death_rate"]
])

## 8. State Growth — May to August

A percentage change is shown only when the starting confirmed count is greater than zero. This avoids division-by-zero errors.

In [ ]:
first_snapshot = (
    analysis.sort_values("date")
    .groupby("state", as_index=False)
    .first()[["state", "date", "confirmed"]]
    .rename(columns={"date": "start_date", "confirmed": "start_confirmed"})
)

last_snapshot = (
    analysis.sort_values("date")
    .groupby("state", as_index=False)
    .last()[["state", "date", "confirmed"]]
    .rename(columns={"date": "end_date", "confirmed": "end_confirmed"})
)

growth = first_snapshot.merge(last_snapshot, on="state")

growth["growth_multiple"] = np.where(
    growth["start_confirmed"] > 0,
    growth["end_confirmed"] / growth["start_confirmed"],
    np.nan
)

growth["percentage_change"] = np.where(
    growth["start_confirmed"] > 0,
    (growth["end_confirmed"] - growth["start_confirmed"])
    / growth["start_confirmed"] * 100,
    np.nan
)

display(growth.sort_values("end_confirmed", ascending=False))

## 9. Visualisations — 10 Revised Charts

### Chart 1 — Selected states: cumulative confirmed cases

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for state in selected_states:
    sub = analysis[analysis["state"] == state]
    ax.plot(sub["date"], sub["confirmed"], label=state, linewidth=2)
ax.set_title("Cumulative Confirmed COVID-19 Cases — Selected States")
ax.set_xlabel("Date")
ax.set_ylabel("Confirmed cases")
ax.legend(fontsize=8, ncol=2)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Chart 2 — Daily new cases across selected states

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(daily["date"], daily["new_cases"], linewidth=2)
ax.set_title("Daily New Confirmed Cases — Selected States")
ax.set_xlabel("Date")
ax.set_ylabel("New cases")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Chart 3 — Confirmed cases on the final date

In [ ]:
plot_df = latest.sort_values("confirmed")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=plot_df, y="state", x="confirmed", hue="state", legend=False, ax=ax)
ax.set_title(f"Confirmed Cases on {latest_date.date()} — Selected States")
ax.set_xlabel("Confirmed cases")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Chart 4 — Active cases on the final date

In [ ]:
plot_df = latest.sort_values("active")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=plot_df, y="state", x="active", hue="state", legend=False, ax=ax)
ax.set_title(f"Active Cases on {latest_date.date()}")
ax.set_xlabel("Active cases")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Chart 5 — Recovery rate comparison

In [ ]:
plot_df = latest.dropna(subset=["recovery_rate"]).sort_values("recovery_rate")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=plot_df, y="state", x="recovery_rate", hue="state", legend=False, ax=ax)
ax.set_title(f"Recovery Rate on {latest_date.date()}")
ax.set_xlabel("Recovery rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Chart 6 — Death rate comparison

In [ ]:
plot_df = latest.dropna(subset=["death_rate"]).sort_values("death_rate")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=plot_df, y="state", x="death_rate", hue="state", legend=False, ax=ax)
ax.set_title(f"Death Rate on {latest_date.date()}")
ax.set_xlabel("Death rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Chart 7 — Monthly confirmed cases by selected state

In [ ]:
analysis["month"] = analysis["date"].dt.to_period("M").astype(str)
monthly = analysis.groupby(["state", "month"], as_index=False)["new_cases"].sum()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=monthly, x="month", y="new_cases", hue="state", ax=ax)
ax.set_title("Monthly Reported New Cases — Selected States")
ax.set_xlabel("Month")
ax.set_ylabel("New cases")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Chart 8 — Start vs end confirmed cases

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=growth, x="start_confirmed", y="end_confirmed",
    size="end_confirmed", sizes=(50, 500), legend=False, ax=ax
)
for _, r in growth.iterrows():
    ax.annotate(r["state"], (r["start_confirmed"], r["end_confirmed"]),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("Confirmed Cases: Start of Period vs End of Period")
ax.set_xlabel("Confirmed cases near 1 May 2020")
ax.set_ylabel("Confirmed cases on 6 August 2020")
plt.tight_layout()
plt.show()

### Chart 9 — Correlation of selected-state metrics

In [ ]:
corr_cols = ["confirmed", "deaths", "cured", "active"]
corr = latest[corr_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation of Final-Date State Metrics")
plt.tight_layout()
plt.show()

### Chart 10 — Distribution of daily new cases

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(daily["new_cases"], bins=20, kde=True, ax=ax)
ax.set_title("Distribution of Daily New Cases — Selected States")
ax.set_xlabel("Daily new cases")
ax.set_ylabel("Number of days")
plt.tight_layout()
plt.show()

## 10. Data-Quality Checks

These checks make the revised analysis safer than the original version.

In [ ]:
checks = {
    "Duplicate (date, state) keys": int(cases.duplicated(["date", "state"]).sum()),
    "Missing dates": int(cases["date"].isna().sum()),
    "Missing states": int(cases["state"].isna().sum()),
    "Negative confirmed": int((cases["confirmed"] < 0).sum()),
    "Negative deaths": int((cases["deaths"] < 0).sum()),
    "Negative recovered": int((cases["cured"] < 0).sum()),
    "Negative active after clipping": int((cases["active"] < 0).sum())
}

for k, v in checks.items():
    print(f"{k}: {v}")

## 11. Save the Revised Clean Dataset

In [ ]:
analysis.to_csv("cleaned_covid_india_revised_period.csv", index=False)
latest.to_csv("covid_india_selected_states_latest_snapshot.csv", index=False)
growth.to_csv("covid_india_selected_states_growth.csv", index=False)

print("Saved:")
print("- cleaned_covid_india_revised_period.csv")
print("- covid_india_selected_states_latest_snapshot.csv")
print("- covid_india_selected_states_growth.csv")

## 12. Findings

The following findings are generated from the selected states and the revised time window, so they remain consistent if the source data changes slightly.

### Key observations
- The analysis covers **1 May 2020 to 6 August 2020**.
- It compares **8 selected states**, rather than ranking every state/UT.
- The final-date snapshot shows how confirmed, active, recovered and death counts differed across the selected states.
- Monthly aggregation highlights when reported new cases accelerated within each selected state.
- Recovery and death rates are presented as **descriptive ratios**, not as measures of individual clinical outcomes.
- The revised analysis avoids calling any state "least affected" because confirmed case counts alone are not directly comparable without population, testing and reporting context.

### Important limitation
This is historical reported surveillance data. Differences between states can reflect testing capacity, reporting practices, population size, healthcare access, timing of outbreaks and other factors. Therefore, the charts should be interpreted as descriptive EDA rather than causal epidemiological conclusions.